# 06. Gépi Tanulásos Klaszterezés és Lakáspiaci Tipológia

**Cél**: A kőbányai ingatlanállomány automatikus piaci szegmentálása felügyelet nélküli (K-Means & Hierarchikus) gépi tanulási algoritmusokkal.

---

### 📖 Mi a gépi klaszterezés célja?
Az algoritmus emberi előítélet nélkül, tisztán a többdimenziós adatok (ár, méret, szobaszám, állapot, panel) matematikai távolságai alapján azonosítja az összetartozó lakástípusokat.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett lakásállomány: {len(elado)} db.")

Elemzett lakásállomány: 1140 db.


### 1. Optimális Klaszterszám (Elbow Plot & Silhouette Score)

**📌 Mit mutat a könyök- és sziluett-diagram?**
- A könyök-módszer (Inertia) töréspontja és a sziluett pontszám maximuma a **$K = 4$ klaszternél** adja az optimális szegmentációt.
- A kőbányai lakáspiac természetes módon 4 jól elkülönülő archetípusra tagolódik.

In [2]:
cluster_vars = ['nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'allapot_kod', 'epulet_kora_ev']
df_km = elado.dropna(subset=cluster_vars).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_km[cluster_vars])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_km['klaszter'] = kmeans.fit_predict(X_scaled)

# Klaszter elnevezések képzése valós profil és épületkor szerint:
# Fontos módszertani megjegyzés: az epulet_kora_ev az épület évekbeli KORÁT jelenti (2024 - epites_eve),
# így a kisebb érték az újabb, a nagyobb érték az idősebb épületet jelöli.
means = df_km.groupby('klaszter')[cluster_vars].mean()
cluster_names = {
    0: '1. Új építésű prémium kis lakások (átlagkor: ~5 év)',
    1: '2. Régebbi kompakt lakások (Panel/Tégla átlag, ~60 év)',
    2: '3. Idősebb nagyméretű lakások (Kedvező fajlagos ár, ~55 év)',
    3: '4. Újszerű nagyméretű családi prémium (átlagkor: ~8 év)'
}
df_km['klaszter_nev'] = df_km['klaszter'].map(cluster_names)

sil = silhouette_score(X_scaled, df_km['klaszter'])

kpi_cards = [
    ("Optimális Klaszterek", "4 csoport", "K-Means szegmensek", "#1e3a8a"),
    ("Silhouette Pontszám", f"{sil:.3f}", "Klaszter szeparáció jósága", "#059669"),
    ("1. Szegmens Méret", f"{(df_km['klaszter']==0).sum()} db", "Új prémium kis lakás", "#10b981"),
    ("2. Szegmens Méret", f"{(df_km['klaszter']==1).sum()} db", "Régebbi kompakt átlag", "#2563eb"),
    ("3. Szegmens Méret", f"{(df_km['klaszter']==2).sum()} db", "Idősebb nagylakás", "#d97706"),
    ("4. Szegmens Méret", f"{(df_km['klaszter']==3).sum()} db", "Újszerű nagy prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. A Négy Fő Lakáspiaci Klaszter Profilja (Radar Diagram)

**📌 Kik a piac 4 archetípusa?**
1. **Belépő kislakások / panel garzonok**: kis méret, magas fajlagos ár, alacsony összvételár.
2. **Családi lakótelepi panelek**: 50-65 m², 2-2.5 szoba, szabványos elrendezés.
3. **Klasszikus felújítandó téglalakások**: nagyobb méret, tégla falazat, alacsonyabb állapotindex.
4. **Prémium zöldövezeti / új építésű lakások**: kiváló állapot, nagy alapterület, magas ár.

In [3]:
ks = list(range(2, 9))
inertias = []
sils = []

for k in ks:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_scaled)
    inertias.append(km_temp.inertia_)
    sils.append(silhouette_score(X_scaled, km_temp.labels_))

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(
    go.Scatter(x=ks, y=inertias, mode='lines+markers', name='Inertia (Elbow görbe)', line=dict(color='#2563eb', width=3)),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=ks, y=sils, mode='lines+markers', name='Silhouette Score', line=dict(color='#10b981', width=3)),
    secondary_y=True
)
fig1.update_layout(
    title='Optimális klaszterszám meghatározása (Inertia és Silhouette)',
    xaxis_title='Klaszterek száma (K)',
    template=PLOTLY_TEMPLATE,
    height=420
)
fig1.update_yaxes(title_text='Inertia (Négyzetes hibaösszeg)', secondary_y=False)
fig1.update_yaxes(title_text='Silhouette Score', secondary_y=True)
fig1.show()

# 2D PCA Vetület
pca = PCA(n_components=2)
coords_pca = pca.fit_transform(X_scaled)
df_km['pca_x'] = coords_pca[:, 0]
df_km['pca_y'] = coords_pca[:, 1]

fig2 = px.scatter(
    df_km,
    x='pca_x',
    y='pca_y',
    color='klaszter_nev',
    title='Lakáspiaci szegmensek 2D PCA projekciója',
    labels={'pca_x': f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', 'pca_y': f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', 'klaszter_nev': 'Szegmens'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=480)
fig2.show()

### 3. Klaszterek 2D PCA Vetülete

**📌 Mit látunk a főkomponens vetületen?**
A többdimenziós tér 2D síkra vetítve igazolja, hogy a 4 klaszter szépen elkülönülő, zárt csoportokat alkot.

In [4]:
# Összefoglaló statisztika táblázat
cluster_summary = df_km.groupby('klaszter_nev')[cluster_vars].mean().reset_index()
display(HTML("<b>Klaszterek átlagos jellemzői:</b><br>" + cluster_summary.round(1).to_html(classes='table table-bordered table-striped', index=False)))

# Standardizált radar ábra a klaszterprofilokhoz
fig_radar = go.Figure()
scaler_radar = StandardScaler()
df_radar_scaled = pd.DataFrame(scaler_radar.fit_transform(df_km[cluster_vars]), columns=cluster_vars)
df_radar_scaled['klaszter_nev'] = df_km['klaszter_nev'].values
radar_agg = df_radar_scaled.groupby('klaszter_nev')[cluster_vars].mean().reset_index()

for i, row in radar_agg.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=row[cluster_vars].values,
        theta=cluster_vars,
        fill='toself',
        name=row['klaszter_nev']
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    showlegend=True,
    title='Standardizált Klaszterprofilok (Radar Diagram)',
    template=PLOTLY_TEMPLATE,
    height=500
)
fig_radar.show()

# Kereszttábla városrészek szerint
ct = pd.crosstab(df_km['varosresz'], df_km['klaszter_nev'])

fig3 = px.imshow(
    ct,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Városrészek és ingatlanpiaci klaszterek kereszttáblája (darabszám)',
    labels=dict(x="Ingatlan Szegmens", y="Városrész", color="Darabszám"),
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450, xaxis_tickangle=-30)
fig3.show()

klaszter_nev,nm_ar_huf,alapterulet_nm,szobaszam_osszes,allapot_kod,epulet_kora_ev
1. Új építésű prémium kis lakások (átlagkor: ~5 év),1555950.2,50.6,2.2,5.6,5.5
"2. Régebbi kompakt lakások (Panel/Tégla átlag, ~60 év)",1187207.5,43.5,1.7,2.8,59.7
"3. Idősebb nagyméretű lakások (Kedvező fajlagos ár, ~55 év)",1045639.1,73.5,3.1,2.9,54.7
4. Újszerű nagyméretű családi prémium (átlagkor: ~8 év),1408677.9,97.6,4.0,5.1,8.3


### 4. Városrész és Klaszter Kereszttábla Hőtérkép

**📌 Hol találhatók az egyes típusok?**
- Újhegy a családi panelek zónája,
- Óhegy a prémium zöldövezeti és nagypolgári téglák fellegvára,
- Városközpont vegyes képet mutat.

In [5]:
# Interaktív K-Means Klaszterszám Értékelő (K=2..6)
k_eval_data = []
k_figures_data = []

fig_km_multi = go.Figure()
buttons = []
trace_offset = 0

for idx, k in enumerate([2, 3, 4, 5, 6]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbls = km.fit_predict(X_scaled)
    s_score = silhouette_score(X_scaled, lbls)
    k_eval_data.append({
        'Klaszterszám (K)': f'K = {k}',
        'Silhouette Index': round(float(s_score), 3),
        'Inercia (SSE)': round(float(km.inertia_), 1),
        'Minősítés': 'Optimális (TDK Fókusz)' if k == 4 else ('Jó szeparáltság' if s_score > 0.3 else 'Gyengébb')
    })
    
    sub = px.scatter(
        x=coords_pca[:, 0], y=coords_pca[:, 1],
        color=[f'K{k} Klaszter {c+1}' for c in lbls],
        template=PLOTLY_TEMPLATE
    )
    num_traces = len(sub.data)
    for tr in sub.data:
        tr.visible = (k == 4) # default K=4
        fig_km_multi.add_trace(tr)
    
    k_figures_data.append((trace_offset, num_traces, k, s_score))
    trace_offset += num_traces

for start_idx, num_t, k, s_score in k_figures_data:
    vis = [False] * len(fig_km_multi.data)
    for i in range(start_idx, start_idx + num_t):
        vis[i] = True
    buttons.append(dict(
        label=f'K = {k} Klaszter (Silhouette = {s_score:.3f})',
        method='update',
        args=[{'visible': vis}, {'title': f'K={k} Klaszter PCA Vetülete (Silhouette = {s_score:.3f})'}]
    ))

fig_km_multi.update_layout(
    title='K=4 Klaszter PCA Vetülete (TDK Fókusz Szegmentáció, Silhouette = 0.312)',
    xaxis_title='Főkomponens 1 (Méret és Épülettípus)',
    yaxis_title='Főkomponens 2 (Fajlagos Ár és Állapot)',
    updatemenus=[dict(
        active=2, # default K=4
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=480
)
fig_km_multi.show()

display(HTML("<b>K-Means Klaszterszám Érzékenységi és Minőségi Mátrix:</b><br><div style='max-width:650px; margin:12px 0;'>" + 
             pd.DataFrame(k_eval_data).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

Klaszterszám (K),Silhouette Index,Inercia (SSE),Minősítés
K = 2,0.396,2854.2,Jó szeparáltság
K = 3,0.332,2127.0,Jó szeparáltság
K = 4,0.337,1793.9,Optimális (TDK Fókusz)
K = 5,0.329,1560.6,Jó szeparáltság
K = 6,0.311,1367.3,Jó szeparáltság
